In [1]:
import pandas as pd
import numpy as np
import json
import time

from pathlib import Path

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

In [2]:
DATA_DIR = Path("../data")
RESULTS_DIR = Path("../results")

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(
    DATA_DIR / "PhiUSIIL_Preprocessed.csv"
)

with open(
    RESULTS_DIR / "feature_config.json",
    "r"
) as f:
    config = json.load(f)

full_features = config["full_features"]
url_only_features = config["url_only_features"]

print("Jumlah sampel:", len(df))
print("Full features:", len(full_features))
print("URL-only features:", len(url_only_features))

Jumlah sampel: 235370
Full features: 47
URL-only features: 19


In [3]:
random_split = np.load(
    RESULTS_DIR / "baseline_split.npz"
)

grouped_split = np.load(
    RESULTS_DIR / "domain_grouped_split.npz"
)

splits = {
    "Random Split": {
        "train_idx": random_split["train_idx"],
        "test_idx": random_split["test_idx"]
    },

    "Domain-Grouped Split": {
        "train_idx": grouped_split["train_idx"],
        "test_idx": grouped_split["test_idx"]
    }
}

print("Kedua strategi split berhasil dimuat.")

Kedua strategi split berhasil dimuat.


In [4]:
y = df["label"].map({
    0: 1,
    1: 0
})

assert not y.isna().any()

print("Distribusi kelas:")
print(y.value_counts())

Distribusi kelas:
label
0    134850
1    100520
Name: count, dtype: int64


In [5]:
def get_models():

    return {
        "Logistic Regression": Pipeline([
            ("scaler", StandardScaler()),
            ("classifier", LogisticRegression(
                max_iter=1000,
                random_state=42
            ))
        ]),

        "Random Forest": RandomForestClassifier(
            n_estimators=100,
            random_state=42,
            n_jobs=-1
        )
    }

In [6]:
def evaluate_model(
    scenario,
    feature_type,
    split_strategy,
    model_name,
    model,
    X_train,
    X_test,
    y_train,
    y_test
):

    start_train = time.perf_counter()

    model.fit(X_train, y_train)

    training_time = time.perf_counter() - start_train

    start_inference = time.perf_counter()

    y_pred = model.predict(X_test)

    inference_time = time.perf_counter() - start_inference

    y_prob = model.predict_proba(X_test)[:, 1]

    tn, fp, fn, tp = confusion_matrix(
        y_test,
        y_pred,
        labels=[0, 1]
    ).ravel()

    return {
        "Scenario": scenario,
        "Feature Type": feature_type,
        "Validation": split_strategy,
        "Model": model_name,
        "Number of Features": X_train.shape[1],
        "Training Samples": len(X_train),
        "Testing Samples": len(X_test),
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(
            y_test, y_pred, zero_division=0
        ),
        "Recall": recall_score(
            y_test, y_pred, zero_division=0
        ),
        "F1-Score": f1_score(
            y_test, y_pred, zero_division=0
        ),
        "ROC-AUC": roc_auc_score(y_test, y_prob),
        "Training Time (s)": training_time,
        "Inference Time (s)": inference_time,
        "Inference per Sample (ms)": (
            inference_time / len(X_test)
        ) * 1000,
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp,
        "Random State": 42
    }

In [7]:
scenarios = {
    "B": {
        "features": url_only_features,
        "feature_type": "URL-Only",
        "split": "Random Split"
    },

    "C": {
        "features": url_only_features,
        "feature_type": "URL-Only",
        "split": "Domain-Grouped Split"
    },

    "D": {
        "features": full_features,
        "feature_type": "Full Features",
        "split": "Domain-Grouped Split"
    }
}

In [8]:
experiment_results = []

for scenario, setup in scenarios.items():

    print("=" * 60)
    print("Scenario:", scenario)
    print("Features:", setup["feature_type"])
    print("Split:", setup["split"])
    print("=" * 60)

    feature_columns = setup["features"]

    split_info = splits[setup["split"]]

    train_idx = split_info["train_idx"]
    test_idx = split_info["test_idx"]

    X_train = df.iloc[train_idx][feature_columns]
    X_test = df.iloc[test_idx][feature_columns]

    y_train = y.iloc[train_idx]
    y_test = y.iloc[test_idx]

    for model_name, model in get_models().items():

        print("Training:", model_name)

        result = evaluate_model(
            scenario,
            setup["feature_type"],
            setup["split"],
            model_name,
            model,
            X_train,
            X_test,
            y_train,
            y_test
        )

        experiment_results.append(result)

        print(
            f"F1-Score: {result['F1-Score']:.6f}"
        )

        print(
            f"ROC-AUC: {result['ROC-AUC']:.6f}"
        )

        print("-" * 40)

Scenario: B
Features: URL-Only
Split: Random Split
Training: Logistic Regression
F1-Score: 0.995439
ROC-AUC: 0.998426
----------------------------------------
Training: Random Forest
F1-Score: 0.997025
ROC-AUC: 0.998226
----------------------------------------
Scenario: C
Features: URL-Only
Split: Domain-Grouped Split
Training: Logistic Regression
F1-Score: 0.995637
ROC-AUC: 0.998484
----------------------------------------
Training: Random Forest
F1-Score: 0.997008
ROC-AUC: 0.998490
----------------------------------------
Scenario: D
Features: Full Features
Split: Domain-Grouped Split
Training: Logistic Regression
F1-Score: 0.999093
ROC-AUC: 0.999980
----------------------------------------
Training: Random Forest
F1-Score: 0.999832
ROC-AUC: 0.999999
----------------------------------------


In [9]:
experiment_df = pd.DataFrame(
    experiment_results
)

display(
    experiment_df[
        [
            "Scenario",
            "Model",
            "Accuracy",
            "Precision",
            "Recall",
            "F1-Score",
            "ROC-AUC"
        ]
    ].round(6)
)

,Scenario,Model,Accuracy,Precision,Recall,F1-Score,ROC-AUC
0,B,Logistic Regression,0.996120,0.999465,0.991444,0.995439,0.998426
1,B,Random Forest,0.997465,0.999334,0.994727,0.997025,0.998226
2,C,Logistic Regression,0.996312,0.999628,0.991677,0.995637,0.998484
3,C,Random Forest,0.997465,0.998687,0.995335,0.997008,0.998490
4,D,Logistic Regression,0.999231,0.999530,0.998658,0.999093,0.999980
5,D,Random Forest,0.999858,0.999966,0.999698,0.999832,0.999999


In [10]:
experiment_df.to_csv(
    RESULTS_DIR / "experiment_metrics.csv",
    index=False
)

print("Hasil eksperimen berhasil disimpan.")

Hasil eksperimen berhasil disimpan.


In [11]:
baseline_df = pd.read_csv(
    RESULTS_DIR / "baseline_metrics.csv"
)

all_results = pd.concat(
    [
        baseline_df,
        experiment_df
    ],
    ignore_index=True
)

display(
    all_results[
        [
            "Scenario",
            "Model",
            "Feature Type",
            "Validation",
            "Accuracy",
            "Precision",
            "Recall",
            "F1-Score",
            "ROC-AUC"
        ]
    ].round(6)
)

,Scenario,Model,Feature Type,Validation,Accuracy,Precision,Recall,F1-Score,ROC-AUC
0,A,Logistic Regression,Full Features,Random Split,0.999278,0.999768,0.998541,0.999154,0.999964
1,A,Random Forest,Full Features,Random Split,0.999830,0.999867,0.999735,0.999801,0.999998
2,B,Logistic Regression,URL-Only,Random Split,0.996120,0.999465,0.991444,0.995439,0.998426
3,B,Random Forest,URL-Only,Random Split,0.997465,0.999334,0.994727,0.997025,0.998226
4,C,Logistic Regression,URL-Only,Domain-Grouped Split,0.996312,0.999628,0.991677,0.995637,0.998484
5,C,Random Forest,URL-Only,Domain-Grouped Split,0.997465,0.998687,0.995335,0.997008,0.998490
6,D,Logistic Regression,Full Features,Domain-Grouped Split,0.999231,0.999530,0.998658,0.999093,0.999980
7,D,Random Forest,Full Features,Domain-Grouped Split,0.999858,0.999966,0.999698,0.999832,0.999999


In [12]:
all_results.to_csv(
    RESULTS_DIR / "all_scenario_metrics.csv",
    index=False
)

print("Seluruh hasil skenario A-D berhasil disimpan.")

Seluruh hasil skenario A-D berhasil disimpan.


Berdasarkan hasil eksperimen, terdapat tiga temuan awal.

Pertama, pembatasan fitur menjadi URL-only memiliki pengaruh yang lebih besar terhadap F1-score daripada perubahan strategi pembagian data.

Penurunan F1-score akibat pembatasan fitur berkisar 0,28–0,37 poin persentase.

Sebaliknya, perubahan F1-score antara random split dan domain-grouped split berada pada kisaran yang jauh lebih kecil.

Kedua, Random Forest menghasilkan F1-score lebih tinggi daripada Logistic Regression pada seluruh skenario.

Namun, kita tetap harus membandingkan waktu pelatihan dan inferensi sebelum menyimpulkan model mana yang paling efisien.

Ketiga, domain-grouped split tidak menunjukkan penurunan performa yang konsisten.

Ini menunjukkan bahwa dugaan mengenai overestimasi performa akibat random split belum terbukti melalui hasil eksperimen tunggal ini.

Kita masih perlu menguji kestabilan hasil menggunakan beberapa random seed dan melakukan pemeriksaan lebih lanjut terhadap fitur dataset.